In [ ]:
from astroquery.simbad import Simbad
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table
import numpy as np

# Reset Simbad to default fields to avoid conflicts
Simbad.reset_votable_fields()

# Add the necessary fields using the new names
Simbad.add_votable_fields("V")  # V-band magnitude
Simbad.add_votable_fields("B")  # B-band magnitude (optional)
Simbad.add_votable_fields("R")  # R-band magnitude (optional)
Simbad.add_votable_fields("sp_type")  # Spectral type
Simbad.add_votable_fields("mesvar")  # Variability flag
Simbad.add_votable_fields("plx_value")  # Parallax
Simbad.add_votable_fields("otype")  # Object type (to filter stars)

# Define the center of the COSMOS field
ra_center = 150.1191667 * u.deg  # 10h 00m 28.6s
dec_center = 2.2058333 * u.deg  # +02° 12' 21.0"
radius = 0.5 * u.deg  # Reduced radius to avoid too many results

# Coordinates of the center
center = SkyCoord(ra=ra_center, dec=dec_center, frame="icrs")

# Query Simbad for all objects in the COSMOS field
print("Querying Simbad for objects in the COSMOS field...")
result = Simbad.query_region(center, radius=radius)

# Check if results were found
if result is None or len(result) == 0:
    print("No results found for the specified region.")
else:
    print(f"Number of objects found in the region: {len(result)}")

    # Convert to Astropy Table for easier processing
    table = Table(result)

    # Print available column names for debugging
    print("\nAvailable columns in the result:")
    print(table.colnames)

    # Debug: Print unique values in 'otype' to understand its content
    if "otype" in table.colnames:
        unique_otypes = set(table["otype"])
        print(f"\nUnique values in 'otype': {unique_otypes}")

    # Filter for stars only (using 'otype')
    # In Simbad, stars are typically marked with 'otype' values starting with '*'
    # (e.g., '*', '**', 'BS*', 'WD*', 'Em*', etc.)
    if "otype" in table.colnames:
        # Handle both bytes and str types, and check if 'otype' starts with '*'
        star_mask = np.array(
            [
                (isinstance(row, bytes) and row.decode("utf-8").startswith("*"))
                or (isinstance(row, str) and row.startswith("*"))
                for row in table["otype"]
            ]
        )
        table = table[star_mask]
        print(f"Number of stars after filtering by 'otype': {len(table)}")
    else:
        print("Warning: 'otype' column not found. Cannot filter for stars only.")

    # If the table is empty after filtering, stop here
    if len(table) == 0:
        print("No stars found after filtering by 'otype'.")
    else:
        # Filter for V-band magnitude
        if "V" in table.colnames:
            # Filter out stars without V-band magnitude
            # Handle both bytes and masked arrays
            if hasattr(table["V"], "mask"):
                # If V is a masked array, use the mask to filter out invalid values
                valid_V = ~table["V"].mask
            else:
                # Otherwise, check for '--' or b'--'
                if len(table) > 0:
                    if isinstance(table["V"][0], bytes):
                        valid_V = np.array([row.decode("utf-8") != "--" for row in table["V"]])
                    else:
                        valid_V = table["V"] != "--"
                else:
                    valid_V = np.array([])

            if len(valid_V) > 0:
                table = table[valid_V]

            # If the table is empty after filtering, stop here
            if len(table) == 0:
                print("No stars found after filtering by V-band magnitude.")
            else:
                # Convert V-band magnitude to float (for filtering)
                if hasattr(table["V"], "mask"):
                    # If V is a masked array, fill masked values with NaN and convert to float
                    V_values = table["V"].filled(fill_value=np.nan)
                    table["V"] = V_values.astype(float)
                else:
                    if isinstance(table["V"][0], bytes):
                        table["V"] = np.array([float(row.decode("utf-8")) for row in table["V"]])
                    else:
                        table["V"] = table["V"].astype(float)

                # Filter for magnitude between 17 and 20
                table = table[(table["V"] >= 17) & (table["V"] <= 20)]

                # If the table is empty after magnitude filtering, stop here
                if len(table) == 0:
                    print("No stars found after filtering by V-band magnitude range.")
                else:
                    # Add a column to indicate stability
                    # Use 'mesvar.vartyp' to check for variability
                    table["stable"] = "Yes"
                    if "mesvar.vartyp" in table.colnames:
                        # Mark as 'No' if mesvar.vartyp is not '--' or b'--'
                        if len(table) > 0:
                            if isinstance(table["mesvar.vartyp"][0], bytes):
                                is_variable = np.array(
                                    [row.decode("utf-8") != "--" for row in table["mesvar.vartyp"]]
                                )
                            else:
                                is_variable = table["mesvar.vartyp"] != "--"
                            table["stable"][is_variable] = "No"

                    # Rename columns for clarity (optional)
                    if "main_id" in table.colnames:
                        table.rename_column("main_id", "ID")
                    if "ra" in table.colnames:
                        table.rename_column("ra", "RA_deg")
                    if "dec" in table.colnames:
                        table.rename_column("dec", "DEC_deg")
                    if "V" in table.colnames:
                        table.rename_column("V", "V_mag")
                    if "sp_type" in table.colnames:
                        table.rename_column("sp_type", "spectral_type")

                    # Display a preview of the results
                    print(f"\nNumber of stars with V magnitude between 17 and 20: {len(table)}")
                    print("\nPreview of the results:")
                    print(table[:10])

                    # Export results to a CSV file
                    csv_filename = "cosmos_stars_17-20_mag.csv"
                    table.write(csv_filename, format="csv", overwrite=True)
                    print(f"\nResults saved to '{csv_filename}'.")

                    # Quick statistics
                    print(f"\nStatistics:")
                    print(f"- Total number of stars: {len(table)}")
                    print(f"- Number of stable stars: {len(table[table['stable'] == 'Yes'])}")
                    print(f"- Number of variable stars: {len(table[table['stable'] == 'No'])}")

                    # Example of additional processing: count unique spectral types
                    spectral_type_column = "spectral_type" if "spectral_type" in table.colnames else "sp_type"
                    if spectral_type_column in table.colnames:
                        spectral_types = table[spectral_type_column].tolist()
                        unique_types = set(
                            [
                                st.decode("utf-8") if isinstance(st, bytes) else str(st)
                                for st in spectral_types
                                if st != b"--" and st != "--" and st is not None and str(st) != "nan"
                            ]
                        )
                        print(f"\nUnique spectral types found: {unique_types}")